# Exam Score Prediction
Using Feature Engineering with Ridge, XGBoost and LGBM

## Library Loading and Configuration

In [1]:
# ==============================================================================
# LIBRARY LOADING AND CONFIGURATION
# ==============================================================================

import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import TargetEncoder
from sklearn.linear_model import RidgeCV

from scipy.optimize import minimize
import warnings

warnings.filterwarnings("ignore")
np.random.seed(42)
FOLDS = 15

print("="*80)
print("Completed loading Libraries")
print("="*80)

Completed loading Libraries


## Data Loading and Configuration

In [2]:
# ==============================================================================
# DATA LOADING
# ==============================================================================
print("="*80)
print("Loading Data")
print("="*80)

train_df = pd.read_csv("/kaggle/input/playground-series-s6e1/train.csv")
test_df = pd.read_csv("/kaggle/input/playground-series-s6e1/test.csv")
original_df = pd.read_csv("/kaggle/input/exam-score-prediction/Exam_Score_Prediction.csv")



TARGET = "exam_score"
ID_COL = "id"
base_features = [col for col in train_df.columns if col not in [TARGET, ID_COL]]
CATS = train_df.select_dtypes("object").columns.to_list()

print("="*80)
print("Completed loading")
print("="*80)

train_df.head()

Loading Data
Completed loading


,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,easy,78.3
1,1,18,other,diploma,4.95,94.8,yes,4.7,poor,self-study,medium,moderate,46.7
2,2,20,female,b.sc,4.68,92.6,yes,5.8,poor,coaching,high,moderate,99.0
3,3,19,male,b.sc,2.00,49.5,yes,8.3,average,group study,high,moderate,63.9
4,4,23,male,bca,7.65,86.9,yes,9.6,good,self-study,high,easy,100.0


## Feature Engineering

In [3]:
# ==============================================================================
# FEATURE ENGINEERING + STATISTICS
# ==============================================================================
def preprocess_optimized(df):
    df_temp = df.copy()
    LUT = {
        'sleep_quality': {'good': 5, 'average': 0, 'poor': -5},
        'facility_rating': {'high': 4, 'medium': 0, 'low': -4},
        'study_method': {
            'coaching': 10,
            'mixed': 5,
            'group study': 2,
            'online videos': 1,
            'self-study': 0
        }
    }
    numeric_features = []
    df_temp['manual_formula'] = (
        6.0 * df_temp['study_hours'] +
        0.35 * df_temp['class_attendance'] +
        1.5 * df_temp['sleep_hours'] +
        df_temp['sleep_quality'].map(LUT['sleep_quality']) +
        df_temp['study_method'].map(LUT['study_method']) +
        df_temp['facility_rating'].map(LUT['facility_rating'])
    )

    df_temp['feature_formula'] = (
        5.9051 * df_temp['study_hours'] +
        0.3454 * df_temp['class_attendance'] +
        1.4234 * df_temp['sleep_hours'] +
        4.7819
    ).astype("float32")

    numeric_features.extend(['manual_formula', 'feature_formula'])

    df_temp['study_hours_sin'] = np.sin(
        2 * np.pi * df_temp['study_hours'] / 12
    ).astype('float32')

    df_temp['class_attendance_sin'] = np.sin(
        2 * np.pi * df_temp['class_attendance'] / 12
    ).astype('float32')

    numeric_features.extend(['study_hours_sin', 'class_attendance_sin'])

    for col in base_features:
        if col in CATS:
            continue

        df_temp[f'log_{col}'] = np.log1p(df_temp[col])
        df_temp[f'{col}_sq'] = df_temp[col] ** 2

        numeric_features.extend([f'log_{col}', f'{col}_sq'])

    final_features = base_features + numeric_features

    return df_temp[final_features], numeric_features

X_raw, numeric_cols = preprocess_optimized(train_df)
y = train_df[TARGET].reset_index(drop=True)

X_test_raw, _ = preprocess_optimized(test_df)

X_orig_raw, _ = preprocess_optimized(original_df)
y_orig = original_df[TARGET].reset_index(drop=True)

print("="*80)
print("Completed computing Feature Engineering")
print("="*80)

Completed computing Feature Engineering


In [4]:
# ==============================================================================
# FEATURE ENGINEERING STATISTICS
# ==============================================================================
print("\n" + "-"*80)
print("FEATURE ENGINEERING STATISTICS")
print("-"*80)

n_base = len(base_features)
n_engineered = len(numeric_cols)
n_total = n_base + n_engineered

print(f"Base features:               {n_base}")
print(f"Engineered features added:   {n_engineered}")
print(f"Total features after FE:     {n_total}")

print("\nBreakdown of engineered features:")

print(f"  • Manual + formula features: 2")
print(f"  • Cyclical (sin) features:   2")

# Log + square features
log_sq_features = [
    f for f in numeric_cols
    if f.startswith("log_") or f.endswith("_sq")
]

print(f"  • Log + squared features:    {len(log_sq_features)}")

print("=" * 80)



--------------------------------------------------------------------------------
FEATURE ENGINEERING STATISTICS
--------------------------------------------------------------------------------
Base features:               11
Engineered features added:   12
Total features after FE:     23

Breakdown of engineered features:
  • Manual + formula features: 2
  • Cyclical (sin) features:   2
  • Log + squared features:    8


## Configuration and Processing

In [5]:
# Processing for Categorical Consistency
full_data = pd.concat([X_raw, X_test_raw, X_orig_raw], axis=0, ignore_index=True)
for col in base_features:
    if col in CATS:
        full_data[col] = full_data[col].astype(str).astype("category")
for col in numeric_cols:
    full_data[col] = full_data[col].astype(float)

X = full_data.iloc[:len(train_df)].copy()
X_test = full_data.iloc[len(train_df):len(train_df) + len(test_df)].copy()
X_original = full_data.iloc[len(train_df) + len(test_df):].copy()

# Stratification for Regression
MIN, MAX = y.min(), y.max()
y_skf = np.select([(y <= MIN), (y > MIN) & (y < MAX), (y >= MAX)], [0, 1, 2])

print("="*80)
print("Completed processing for Categorical Consistency")
print("="*80)

Completed processing for Categorical Consistency


## Model Parameters

In [6]:
# ==============================================================================
# MODEL PARAMETERS
# ==============================================================================
xgb_params = {
    'n_estimators': 4000, 
    'learning_rate': 0.00125,
    'max_depth': 9, 
    'subsample': 0.60, 
    'colsample_bytree': 0.85, 
    'colsample_bynode': 0.25, 
    'min_child_weight': 9,
    "tree_method": "hist",
    "random_state": 42,
    "early_stopping_rounds": 100,
    "eval_metric": "rmse",
    "enable_categorical": True,
    "device": "cuda"
}

lgb_params = {
    "objective": "regression",
    "metric": "rmse",
    "n_estimators": 6000,       
    "learning_rate": 0.01,     
    "max_depth": 8,   
    "num_leaves": 63,   
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.7,
    "min_child_samples": 50,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": 42,
    "verbosity": -1
}

print("="*80)
print("Completed Model Parameters")
print("="*80)

Completed Model Parameters


## Cross Validation

In [7]:
# ==============================================================================
# CROSS-VALIDATION LOOP (XGB + LGBM)
# ==============================================================================
kf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=1003)

oof_xgb = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
preds_xgb = []
preds_lgb = []

print(f"\n{'='*80}")
print("TRAINING MODELS")
print("="*80)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_skf), start=1):
    X_tr, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Add Original Data to Training
    X_tr_comb = pd.concat([X_tr, X_original], axis=0)
    y_tr_comb = pd.concat([y_tr, y_orig], axis=0)

    # Ridge Meta-Feature
    te = TargetEncoder(smooth='auto', target_type='continuous')
    X_tr_enc = X_tr_comb.copy()
    X_val_enc = X_val.copy()
    X_te_enc = X_test.copy()
    
    X_tr_enc[CATS] = te.fit_transform(X_tr_comb[CATS], y_tr_comb)
    X_val_enc[CATS] = te.transform(X_val[CATS])
    X_te_enc[CATS] = te.transform(X_test[CATS])
    
    # Remove category dtypes for Ridge
    for df_enc in [X_tr_enc, X_val_enc, X_te_enc]:
        for col in CATS: df_enc[col] = df_enc[col].astype(float)

    ridge = RidgeCV(alphas=np.logspace(-3, 3, 10))
    ridge.fit(X_tr_enc, y_tr_comb)
    
    # Add Ridge prediction back as a feature
    X_tr_comb['feature_lr_pred'] = ridge.predict(X_tr_enc)
    X_val['feature_lr_pred'] = ridge.predict(X_val_enc)
    X_te_final = X_test.copy()
    X_te_final['feature_lr_pred'] = ridge.predict(X_te_enc)

    # --- XGBoost ---
    m_xgb = xgb.XGBRegressor(**xgb_params)
    m_xgb.fit(X_tr_comb, y_tr_comb, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = m_xgb.predict(X_val)
    preds_xgb.append(m_xgb.predict(X_te_final))

    # --- LightGBM ---
    m_lgb = lgb.LGBMRegressor(**lgb_params)
    m_lgb.fit(X_tr_comb, y_tr_comb, eval_set=[(X_val, y_val)], 
              callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    oof_lgb[val_idx] = m_lgb.predict(X_val)
    preds_lgb.append(m_lgb.predict(X_te_final))
    
    print(f"Fold {fold:2d} | XGB RMSE: {np.sqrt(mean_squared_error(y_val, oof_xgb[val_idx])):.4f} | LGBM RMSE: {np.sqrt(mean_squared_error(y_val, oof_lgb[val_idx])):.4f}")



TRAINING MODELS
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[5998]	valid_0's rmse: 8.64346
Fold  1 | XGB RMSE: 8.7301 | LGBM RMSE: 8.6435
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[6000]	valid_0's rmse: 8.67965
Fold  2 | XGB RMSE: 8.7533 | LGBM RMSE: 8.6796
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[6000]	valid_0's rmse: 8.73049
Fold  3 | XGB RMSE: 8.8117 | LGBM RMSE: 8.7305
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[6000]	valid_0's rmse: 8.72687
Fold  4 | XGB RMSE: 8.7885 | LGBM RMSE: 8.7269
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[5987]	valid_0's rmse: 8.68126
Fold  5 | XGB RMSE: 8.7474 | LGBM RMSE: 8.6813
Training until validation scores don't impr

## Weight Optimzation

In [8]:
# ==============================================================================
# WEIGHT OPTIMIZATION
# ==============================================================================
def objective(weights):
    blend = weights[0] * oof_xgb + (1 - weights[0]) * oof_lgb
    return np.sqrt(mean_squared_error(y, blend))

res = minimize(objective, [0.5], bounds=[(0, 1)], method='Nelder-Mead')
best_weight = res.x[0]

print(f"\n{'='*80}")
print(f"OPTIMIZATION COMPLETE")
print(f"Best XGB Weight: {best_weight:.4f}")
print(f"Best LGBM Weight: {1-best_weight:.4f}")
print(f"Blended OOF RMSE: {res.fun:.5f}")
print("="*80)

# Final Ensemble
final_test_preds = best_weight * np.mean(preds_xgb, axis=0) + (1 - best_weight) * np.mean(preds_lgb, axis=0)




OPTIMIZATION COMPLETE
Best XGB Weight: 0.0000
Best LGBM Weight: 1.0000
Blended OOF RMSE: 8.68861


## Final Submission

In [9]:
#  ==============================================================================
# FINAL SUBMISSION
# ==============================================================================

submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET: final_test_preds
})

submission.to_csv("submission.csv", index=False)
print("submission.csv saved")
print(submission.head())

pd.DataFrame({"xgb_oof": oof_xgb, "lgb_oof": oof_lgb}).to_csv('oofs.csv', index=False)
print("Files saved: submission.csv, oofs.csv")

submission.csv saved
       id  exam_score
0  630000   71.088904
1  630001   70.580234
2  630002   87.538083
3  630003   55.391737
4  630004   46.841434
Files saved: submission.csv, oofs.csv
